# TFM: Análisis de Políticas de Sostenibilidad mediante técnicas de Argumentacion Computacional

## Detección de Argumentos con Gemma 4B

- ollama serve
- ollama run gemma3:4b

In [15]:
#%pip install langchain pymupdf openai openpyxl --quiet



In [16]:
from typing import List
from pydantic import BaseModel, Field, ValidationError
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate
from langchain_core.exceptions import OutputParserException
import requests
import json
import re
from openai import OpenAI
import openai
import httpx
import pandas as pd
import numpy as np
import os
import openpyxl

process_text_path = "..\\Data\\Processed Files (sections)\\"

model_name="gemma3:4b"
prefix = 'GLOBAL_SGD2025_'
output_dir = "..\\Data\\Extracted Arguments No Keywords (all text)\\"

## Input text processing

In [17]:
# 1. Define your Pydantic schema for output
class ArgumentResponse(BaseModel):
    arguments: List[str] = Field(..., description="List of arguments extracted directly from the text.")

# 2. Setup output parser
pydantic_parser = PydanticOutputParser(pydantic_object=ArgumentResponse)

# 3. Extend text with first sentence from the next page
def extend_pages_with_next_sentence(pages):
    def get_first_sentence(text):
        match = re.search(r'(.+?\.)', text.strip())
        return match.group(1).strip() if match else ""

    extended_pages = []
    for i, page in enumerate(pages):
        current_text = page["text"]
        if i + 1 < len(pages):
            next_sentence = get_first_sentence(pages[i + 1]["text"])
            current_text += " " + next_sentence
        extended_pages.append({
            "page": page["page"],
            "text": current_text
        })
    return extended_pages

# 4. Build the prompt and call the LLM to extract arguments
def extract_arguments_json(text, topic, model_name) -> ArgumentResponse:
    format_instructions = pydantic_parser.get_format_instructions()

    prompt = PromptTemplate(
        template=(
            "Task: Text Span Identification for Arguments related to Sustainable Development Goal: {topic}\n\n"
            "Role: You are an expert in logical reasoning, sustainability reporting, and argument analysis. "
            "Your job is to identify and extract **verbatim arguments** about {topic} from long-form sustainability texts.\n\n"
            "Instructions:\n"
            "1. Carefully read the entire input text.\n"
            "2. Identify ONLY those sentences or phrases that:\n"
            "   - Clearly support or argue for or against the topic {topic}\n"
            "   - Contain keyword from the relevant lists below\n"
            "   - Are exclusively about {topic} (EXCLUDE if they mention or refer to other SDGs or unrelated sustainability topics)\n\n"
            "3. Each extracted argument must:\n"
            "   - Relate exclusively to the specified SDG ({topic})\n"
            "   - Stand as a full statement\n"
            "   - Be copied exactly from the original (no paraphrasing)\n"
            "   - Include only the necessary context for understanding\n"
            "4. If no qualifying arguments are found, return an empty array.\n\n"
            "Output Rules:\n"
            "- Use **only the exact text** from the original\n"
            "- Do **not** add or reword anything\n"
            "- Return only valid JSON\n"
            "- No markdown (```), no extra explanation\n\n"
            "Text:\n\"\"\"\n{text}\n\"\"\"\n\n"
            "Respond ONLY with a JSON object like this:\n\n"
            "{format_instructions}"
        ),
        input_variables=["text", "topic"],
        partial_variables={"format_instructions": format_instructions}
    )

    final_prompt = prompt.format_prompt(text=text, topic=topic).to_string()

    payload = {
        "model": model_name,
        "prompt": final_prompt,
        "temperature": 0,
        "stream": False
    }

    response = requests.post("http://localhost:11434/api/generate", json=payload)
    if response.status_code != 200:
        raise Exception(f"Ollama error: {response.text}")

    raw_output = response.json()["response"]
    print("Model Output:", raw_output)

    try:
        return pydantic_parser.parse(raw_output)
    except OutputParserException as err:
        print("Parse failed:", err)
        return ArgumentResponse(arguments=[])

# 5. Wrapper function for pipeline
def extract_arguments_from_text(text, topic, model_name) -> List[str]:
    result = extract_arguments_json(text, topic, model_name)
    return result.arguments

# 6. Main document-level processor
def process_document(pages, model_name, topic=""):
    extended_pages = extend_pages_with_next_sentence(pages)
    processed = []
    for page in extended_pages:
        print(f"\n--- Processing Page {page['page']} ---")
        #print("Text to analyze:\n", page["text"])
        
        arguments = extract_arguments_from_text(page["text"], topic, model_name)
        
        print("Extracted Arguments:")
        for i, arg in enumerate(arguments, 1):
            print(f"{i}. {arg}")

        processed.append({
            "page": page["page"],
            "text": page["text"],
            "arguments": arguments
        })
    return processed


# 7. File I/O
def save_to_json(processed, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(processed, f, indent=2, ensure_ascii=False)

def process_directory(input_dir, output_dir, prefix, model_name, topic="", sgd_number=None):
    os.makedirs(output_dir, exist_ok=True)
    all_results = []

    for filename in os.listdir(input_dir):
        if filename.endswith(".json") and filename.startswith(prefix):
            filepath = os.path.join(input_dir, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                pages = json.load(f)

            section_name = filename.replace(".json", "")
            processed = process_document(pages, model_name, topic)

            for item in processed:
                item["section"] = section_name  # Add section identifier
                all_results.append(item)
                
    return all_results



## SGD 1: Poverty

In [18]:
topic = "SGD 1 (Poverty): End poverty in all its forms everywhere"
sgd_number = "1"
resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Model Output: ```json
{
  "arguments": [
    "Conflicts, structural vulnerabilities, and limited fiscal space impede SDG progress in many parts of the world.",
    "most UN member states have made strong progress on targets related to access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3)"
  ]
}
```
Extracted Arguments:
1. Conflicts, structural vulnerabilities, and limited fiscal space impede SDG progress in many parts of the world.
2. most UN member states have made strong progress on targets related to access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3)

--- Processing Page 11 ---
Model Output: ```json
{"arguments": ["Among G20 countries, Brazil is the most committed

## SGD 2: Hunger

In [19]:
topic = "SGD 2 (Hunger): End hunger, achieve food security and improved nutrition and promote sustainable agriculture"
sgd_number = "2"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Model Output: ```json
{
  "arguments": [
    "access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3)"
  ]
}
```
Extracted Arguments:
1. access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3)

--- Processing Page 11 ---
Model Output: ```json
{
  "arguments": [
    "Among G20 countries, Brazil is the most committed to UN-based multilateralism, with Chile leading among OECD countries.",
    "capital flows in far larger sums to the EMDEs."
  ]
}
```
Extracted Arguments:
1. Among G20 countries, Brazil is the most committed to UN-based multilateralism, with Chile leading among OECD countries.
2. capital flows in far larger sums to the EMDEs.

--- Processing Page 13 ---
Model Ou

## SGD 3: Health

In [20]:
topic = "SGD 3 (Health): Ensure healthy lives and promote well-being for all at all ages"
sgd_number = "3"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Model Output: ```json
{
  "arguments": [
    "under-5 mortality rate (SDG 3)",
    "neonatal mortality (SDG 3)"
  ]
}
```
Extracted Arguments:
1. under-5 mortality rate (SDG 3)
2. neonatal mortality (SDG 3)

--- Processing Page 11 ---
Model Output: ```json
{"arguments": ["Among G20 countries, Brazil is the most committed to UN-based multilateralism, with Chile leading among OECD countries.", "Money flows readily to rich countries and not to the emerging and developing economies (EMDEs) that offer higher growth potential and rates of return.", "Part 1 of this report (also published online by the SDSN in May 2025) offers practical recommendations to scale up and align international financing flows to support global public goods and achieve sustainable development."]
}
```
Extracted Arguments:
1. Among G20 countries, Brazil is the most committed to UN-based multilateralism, with Chile leading among OECD countries.
2. Money flows readily to rich countries and no

## SGD 4: Education

In [21]:
topic = "SGD 4 (Education): Ensure inclusive and equitable quality education"
sgd_number = "4"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Model Output: ```json
{
  "arguments": [
    "mobile broadband use (SDG 9)",
    "access to electricity (SDG 7)",
    "internet use (SDG 9)",
    "under-5 mortality rate (SDG 3)",
    "neonatal mortality (SDG 3)"
  ]
}
```
Extracted Arguments:
1. mobile broadband use (SDG 9)
2. access to electricity (SDG 7)
3. internet use (SDG 9)
4. under-5 mortality rate (SDG 3)
5. neonatal mortality (SDG 3)

--- Processing Page 11 ---
Model Output: ```json
{"arguments": ["Among G20 countries, Brazil is the most committed to UN-based multilateralism, with Chile leading among OECD countries.", "At the top of the agenda at FfD4 is the need to reform the GFA so that capital flows in far larger sums to the EMDEs.", "Part 1 of this report (also published online by the SDSN in May 2025) offers practical recommendations to scale up and align international financing flows to support global public goods and achieve sustainable development."]
}
```
Extracted Arguments:
1. Among G20 

## SGD 5: Gender

In [22]:
topic = "SGD 5 (Gender): Achieve gender equality and empower all women and girls"
sgd_number = "5"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Model Output: ```json
{
  "arguments": [
    "access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3)"
  ]
}
```
Extracted Arguments:
1. access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3)

--- Processing Page 11 ---
Model Output: ```json
{"arguments": ["Among G20 countries, Brazil is the most committed to UN-based multilateralism, with Chile leading among OECD countries."] }
```
Extracted Arguments:
1. Among G20 countries, Brazil is the most committed to UN-based multilateralism, with Chile leading among OECD countries.

--- Processing Page 13 ---
Model Output: ```json
{"arguments": ["The high-income countries have delayed critical capital increases at the World Bank a

## SGD 6: Water and sanitation

In [23]:
topic = "SGD 6 (Water and sanitation): Ensure availability and sustainable management of water and sanitation for all"
sgd_number = "6"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)




--- Processing Page 10 ---
Model Output: ```json
{"arguments": ["access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3)"]}
```
Extracted Arguments:
1. access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3)

--- Processing Page 11 ---
Model Output: ```json
{"arguments": ["Among G20 countries, Brazil is the most committed to UN-based multilateralism, with Chile leading among OECD countries.", "Money flows readily to rich countries and not to the emerging and developing economies (EMDEs) that offer higher growth potential and rates of return.", "Part 1 of this report (also published online by the SDSN in May 2025) offers practical recommendations to scale up and align international financing flows to 

## SGD 7: Clean Energy

In [24]:
topic = "SGD 7 (Clean Energy): Ensure access to affordable, reliable, sustainable and modern energy for all"
sgd_number = "7"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Model Output: ```json
{
  "arguments": [
    "access to electricity (SDG 7)",
    "access to electricity (SDG 7)"
  ]
}
```
Extracted Arguments:
1. access to electricity (SDG 7)
2. access to electricity (SDG 7)

--- Processing Page 11 ---
Model Output: ```json
{"arguments": ["Among G20 countries, Brazil is the most committed to UN-based multilateralism, with Chile leading among OECD countries.", "Money flows readily to rich countries and not to the emerging and developing economies (EMDEs) that offer higher growth potential and rates of return.", "Part 1 of this report (also published online by the SDSN in May 2025) offers practical recommendations to scale up and align international financing flows to support global public goods and achieve sustainable development."]
}
```
Extracted Arguments:
1. Among G20 countries, Brazil is the most committed to UN-based multilateralism, with Chile leading among OECD countries.
2. Money flows readily to rich countries an

## SGD 8: Decent Work, Economic Growth

In [25]:
topic = "SGD 8 (decent work, economic growth): Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all"
sgd_number = "8"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Model Output: ```json
{"arguments": ["None"]}
```
Extracted Arguments:
1. None

--- Processing Page 11 ---
Model Output: ```json
{"arguments": ["Money flows readily to rich countries and not to the emerging and developing economies (EMDEs) that offer higher growth potential and rates of return.", "capital should flow to the emerging and developing countries on more favourable terms.", "capital flows in far larger sums to the EMDEs."] }
```
Extracted Arguments:
1. Money flows readily to rich countries and not to the emerging and developing economies (EMDEs) that offer higher growth potential and rates of return.
2. capital should flow to the emerging and developing countries on more favourable terms.
3. capital flows in far larger sums to the EMDEs.

--- Processing Page 13 ---
Model Output: ```json
{
  "arguments": [
    "The overall cost of UN operations is a paltry sum – just US$46 billion in 2023 (the year of most recent data) compared with US$2.4 trillion

## SGD 9: Infrastructure, industrilization, innovation

In [26]:
topic = "SGD 9 (Infrastructure, industrilization, innovation): Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation"
sgd_number = "9"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Model Output: ```json
{"arguments": ["mobile broadband use (SDG 9)", "internet use (SDG 9)"] }
```
Extracted Arguments:
1. mobile broadband use (SDG 9)
2. internet use (SDG 9)

--- Processing Page 11 ---
Model Output: ```json
{"arguments": ["capital flows in far larger sums to the EMDEs", "capital should flow to the emerging and developing countries on more favourable terms", "sustainable development offers high returns"]}
```
Extracted Arguments:
1. capital flows in far larger sums to the EMDEs
2. capital should flow to the emerging and developing countries on more favourable terms
3. sustainable development offers high returns

--- Processing Page 13 ---
Model Output: ```json
{
  "arguments": [
    "The overall cost of UN operations is a paltry sum – just US$46 billion in 2023 (the year of most recent data) compared with US$2.4 trillion spent worldwide on the military that year.",
    "cutting UN budgets at a time of pervasive conflicts, human displacement

## SGD 10: Inequality

In [27]:
topic = "SGD 10 (Inequality): Reduce inequality within and among countries"
sgd_number = "10"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Model Output: ```json
{
  "arguments": [
    "Conflicts, structural vulnerabilities, and limited fiscal space impede SDG progress in many parts of the world.",
    "most UN member states have made strong progress on targets related to access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3)."
  ]
}
```
Extracted Arguments:
1. Conflicts, structural vulnerabilities, and limited fiscal space impede SDG progress in many parts of the world.
2. most UN member states have made strong progress on targets related to access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3).

--- Processing Page 11 ---
Model Output: ```json
{
  "arguments": [
    "Barbados ranks first and the United Sta

## SGD 11: Sustainable cities

In [28]:
topic = "SGD 11 (Sustainable Cities, Sustainable Communities): Make cities and human settlements inclusive, safe, resilient and sustainable"
sgd_number = "11"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Model Output: ```json
{"arguments": ["access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3)"] , "title": "Arguments", "type": "array"}
```
Extracted Arguments:
1. access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3)

--- Processing Page 11 ---
Model Output: ```json
{"arguments": ["Among G20 countries, Brazil is the most committed to UN-based multilateralism, with Chile leading among OECD countries.", "Money flows readily to rich countries and not to the emerging and developing economies (EMDEs) that offer higher growth potential and rates of return.", "Part 1 of this report (also published online by the SDSN in May 2025) offers practical recommendations to scale up and

## SGD 12: Responsible Consumption, Responsible Production

In [29]:
topic = "SGD 12 (Responsible Consumption, Responsible Production): Ensure sustainable consumption and production patterns"
sgd_number = "12"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Model Output: ```json
{"arguments": ["internet use (SDG 9)", "access to electricity (SDG 7)", "under-5 mortality rate (SDG 3)", "neonatal mortality (SDG 3)"] }
```
Extracted Arguments:
1. internet use (SDG 9)
2. access to electricity (SDG 7)
3. under-5 mortality rate (SDG 3)
4. neonatal mortality (SDG 3)

--- Processing Page 11 ---
Model Output: ```json
{"arguments": ["Among G20 countries, Brazil is the most committed to UN-based multilateralism, with Chile leading among OECD countries.", "Money flows readily to rich countries and not to the emerging and developing economies (EMDEs) that offer higher growth potential and rates of return.", "Part 1 of this report (also published online by the SDSN in May 2025) offers practical recommendations to scale up and align international financing flows to support global public goods and achieve sustainable development."]
}
```
Extracted Arguments:
1. Among G20 countries, Brazil is the most committed to UN-based multil

## SGD 13: Climate change

In [30]:
topic = "SGD 13 (Climate change): Take urgent action to combat climate change and its impacts"
sgd_number = "13"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Model Output: ```json
{"arguments": ["Yet even these countries face significant challenges in achieving at least two goals, including those related to climate and biodiversity.", "Conflicts, structural vulnerabilities, and limited fiscal space impede SDG progress in many parts of the world."]
}
```
Extracted Arguments:
1. Yet even these countries face significant challenges in achieving at least two goals, including those related to climate and biodiversity.
2. Conflicts, structural vulnerabilities, and limited fiscal space impede SDG progress in many parts of the world.

--- Processing Page 11 ---
Model Output: ```json
{
  "arguments": [
    "the United States ranks last in this year’s Index of countries’ support for UN-based multilateralism",
    "the United States announced its withdrawal from the Paris Climate Agreement"
  ]
}
```
Extracted Arguments:
1. the United States ranks last in this year’s Index of countries’ support for UN-based multilateralism


## SGD 14: Life bellow water

In [31]:
topic = "SGD 14 (Life bellow Water): Conserve and sustainably use the oceans, seas and marine resources for sustainable development"
sgd_number = "14"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Model Output: ```json
{
  "arguments": [
    "access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3)"
  ]
}
```
Extracted Arguments:
1. access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3)

--- Processing Page 11 ---
Model Output: ```json
{"arguments": ["Money flows readily to rich countries and not to the emerging and developing economies (EMDEs) that offer higher growth potential and rates of return.", "capital flows in far larger sums to the EMDEs.", "capital flows in far larger sums to the EMDEs."]
}
```
Extracted Arguments:
1. Money flows readily to rich countries and not to the emerging and developing economies (EMDEs) that offer higher growth potential and rates 

## SGD 15: Life on land

In [32]:
topic = "SGD 15 (Life on land): Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss"
sgd_number = "15"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Model Output: ```json
{
  "arguments": [
    "Yet even these countries face significant challenges in achieving at least two goals, including those related to climate and biodiversity.",
    "most UN member states have made strong progress on targets related to access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3)."
  ]
}
```
Extracted Arguments:
1. Yet even these countries face significant challenges in achieving at least two goals, including those related to climate and biodiversity.
2. most UN member states have made strong progress on targets related to access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3).

--- Processing Page 11 ---
Model Output: ```json
{
  "argu

## SGD 16: Peace, Justice, Strong Institutions

In [33]:
topic = "SGD 16 (Peace, Justice, Strong Institutions): Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels"
sgd_number = "16"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Model Output: ```json
{
  "arguments": [
    "Conflicts, structural vulnerabilities, and limited fiscal space impede SDG progress in many parts of the world.",
    "Most UN member states have made strong progress on targets related to access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3)."
  ]
}
```
Extracted Arguments:
1. Conflicts, structural vulnerabilities, and limited fiscal space impede SDG progress in many parts of the world.
2. Most UN member states have made strong progress on targets related to access to basic services and infrastructure, including mobile broadband use (SDG 9), access to electricity (SDG 7), internet use (SDG 9), under-5 mortality rate (SDG 3), and neonatal mortality (SDG 3).

--- Processing Page 11 ---
Model Output: ```json
{
  "arguments": [
    "Barbados stands out as the country most

## SGD 17: Partnerships, sustainable development

In [34]:
topic = "SGD 17 (Partnerships, sustainable development):Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development"
sgd_number = "17"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Model Output: ```json
{"arguments": ["190 out of 193 countries have presented national action plans for advancing sustainable development.", "190 of the 193 UN member states have participated in the Voluntary National Review (VNR) process, presenting their SDG implementation plans and sustainable development priorities to the international community.", "most UN member states have presented two or more VNRs"]}
```
Extracted Arguments:
1. 190 out of 193 countries have presented national action plans for advancing sustainable development.
2. 190 of the 193 UN member states have participated in the Voluntary National Review (VNR) process, presenting their SDG implementation plans and sustainable development priorities to the international community.
3. most UN member states have presented two or more VNRs

--- Processing Page 11 ---
Model Output: ```json
{"arguments": ["UN member states gathering at the 4th International Conference on Financing for Development (

## SGD 0: Overarching terms

In [35]:
topic = "SGD Overarching terms: Sustainable Development Goal, SDG, Agenda 2030, leave no one behind, Voluntary National Review, SDG transformations, "
sgd_number = "0"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 10 ---
Model Output: ```json
{"arguments": ["190 of the 193 UN member states have participated in the Voluntary National Review (VNR) process", "190 out of 193 countries have presented national action plans for advancing sustainable development", "190 of the 193 UN member states have participated in the Voluntary National Review (VNR) process"]}
```
Extracted Arguments:
1. 190 of the 193 UN member states have participated in the Voluntary National Review (VNR) process
2. 190 out of 193 countries have presented national action plans for advancing sustainable development
3. 190 of the 193 UN member states have participated in the Voluntary National Review (VNR) process

--- Processing Page 11 ---
Model Output: ```json
{"arguments": ["The United States ranks last in this year’s Index of countries’ support for UN-based multilateralism.", "Among G20 countries, Brazil is the most committed to UN-based multilateralism, with Chile leading among OECD countries.", "Money flo

In [14]:
!git add .
!git commit -m "sgd 2024 8-17"
!git push origin main  # or 'master' or your branch name

[main a03c53c] sgd 2024 8-17
 1 file changed, 2229 insertions(+), 1975 deletions(-)


error: src refspec # does not match any
error: src refspec or does not match any
error: src refspec 'master' does not match any
error: src refspec or does not match any
error: src refspec your does not match any
error: src refspec branch does not match any
error: src refspec name does not match any
error: failed to push some refs to 'https://github.com/camipalo/TFM.git'
